# Week 2 Day 1 Challenge - the concise wording with the payout tool

On Week 1 Day 4 we wrote a concise version of the policy wording, in `data/policy_wording_short.md`. This notebook runs the agent with the payout tool twice, once with the full wording and once with the concise one, and compares the two experiments in Langfuse:

1. Does the concise wording keep the accuracy?
2. Is it cheaper or faster?
3. When a wrongful decline costs UnderCover more or less than the $1,500 in the lab (`WRONGFUL_DECLINE`), does it change which wording we would choose?

The agent, the payout tool and the scores are the ones from `labs/week2day1.ipynb`. Only the model and the wording change.

## Why Claude Haiku 5.5 and not GPT-6 Luna

The lab calls GPT-6 Luna through OpenRouter. Luna is a paid model there, and my OpenRouter account has no credit: OpenRouter only sells credit by card, with no PayPal, and its crypto and cash payment options don't work in my country. I have credit on Anthropic's API, so this notebook calls Claude Haiku 5.5 directly.

- **The cost is comparable.** Haiku 5.5 costs $0.10 per million input tokens and $0.50 per million output tokens, the same as Luna on OpenRouter. The six runs cost $0.37.
- **The scores aren't comparable with the lab's.** A different model makes different mistakes, so both experiments here run on Haiku, and they're compared with each other, not with the Luna runs.
- **Cost comes from the tokens.** The lab reads each call's cost from OpenRouter's `token_usage.cost`, which only OpenRouter sends. Here it's the tokens LangChain reports times Haiku's prices.
- **What you need:** `ANTHROPIC_API_KEY` in `.env` and the `langchain-anthropic` package (`uv add langchain-anthropic` if your project doesn't have it).

The reasoning effort is `low`, the nearest to the lab's `reasoning_effort="none"` on Luna.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler

# This notebook sits two folders below the repo's root, and the payout rules are in labs/
ROOT = Path("../..")
sys.path.append(str(ROOT / "labs"))
from week2day1_payout import Benefit, Line, Payout, work_out_payout  # noqa: E402

## Connect to Anthropic and Langfuse

Both keys come from the `.env` file in the `undercover` folder.

In [ ]:
load_dotenv(ROOT / ".env", override=True)

if os.getenv("ANTHROPIC_API_KEY"):
    print("Anthropic API key found")
else:
    print("No Anthropic API key found. Add ANTHROPIC_API_KEY to the .env file")

langfuse = get_client()

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

## The harder claims and the two wordings

The same 25 harder claims as the lab, with the handlers' decisions. The concise wording keeps the sections and their rules, in fewer words.

In [ ]:
HARDER = ROOT / "data" / "evals" / "harder"


def load(name):
    """Read one of the JSON files of the harder claims."""
    return json.loads((HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

WORDINGS = {
    "full": (ROOT / "data" / "policy_wording.md").read_text(encoding="utf-8"),
    "concise": (ROOT / "data" / "policy_wording_short.md").read_text(encoding="utf-8"),
}

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
for wording, text in WORDINGS.items():
    print(f"The {wording} wording: {len(text):,} characters")

## The agent with the payout tool

The instructions, the prompt and the two tools are the lab's, unchanged.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

One agent for each wording, on Haiku 5.5. The prices are Anthropic's, in US dollars per million tokens.

In [ ]:
MODEL = "claude-haiku-5-5"
INPUT_PRICE = 0.10
OUTPUT_PRICE = 0.50

haiku = ChatAnthropic(model=MODEL, reasoning_effort="low", max_tokens=16000, timeout=120, max_retries=5)

agents = {
    wording: create_agent(model=haiku, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{text}\n")
    for wording, text in WORDINGS.items()
}

## The scores

The lab's scores. `mistake_cost` takes the cost of a wrongful decline as an argument, so the runs can be scored again with another value later without calling the model.

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected, wrongful_decline=WRONGFUL_DECLINE):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return wrongful_decline
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()

## The dataset

The lab's `harder-claims` dataset. If you ran the lab it's already in Langfuse, and this updates its items in place.

Set `TINY = True` for a quick check on 5 claims. `N_TIMES` is the number of runs of each experiment: the answers vary from run to run, and Ed uses 3. `MAX_CONCURRENCY` is how many claims run at once. The lab runs one at a time; 5 at once makes each run a few times faster.

In [ ]:
DATASET = "harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"]},
        metadata={"tiny": label["tiny"]},
    )

In [ ]:
TINY = False
N_TIMES = 3
MAX_CONCURRENCY = 5

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims, {N_TIMES} times for each wording")

## The task and the evaluators

The task runs an agent on one claim, as in the lab, and also keeps the tokens and the time it took. If the agent finishes without recording a decision, the claim would go to a handler, so it counts as a referral.

In [ ]:
def decide(agent, claim):
    """Run an agent on one claim and return its decision, with the tokens, cost and time of its model calls."""
    started = time.perf_counter()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [CallbackHandler()]},
    )
    seconds = time.perf_counter() - started
    usage = [message.usage_metadata or {} for message in result["messages"] if message.type == "ai"]
    input_tokens = sum(u.get("input_tokens", 0) for u in usage)
    output_tokens = sum(u.get("output_tokens", 0) for u in usage)
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "input_tokens": input_tokens, "output_tokens": output_tokens, "cost": cost, "seconds": seconds}

In [ ]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, and the cost of its mistake."""
    return [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "input_tokens": answer["input_tokens"],
            "llm_cost": answer["cost"],
            "seconds": answer["seconds"],
            "rationale": answer["rationale"],
            "expected": expected,
            "answer": answer,
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "llm_cost": table["llm_cost"].sum(),
        "input_tokens_a_claim": table["input_tokens"].mean(),
        "seconds_a_claim": table["seconds"].mean(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]

## The experiments

Each wording runs `N_TIMES` times. Each run is an experiment on the `harder-claims` dataset in Langfuse, named after the wording and the model, so it sits beside the lab's runs.

In [ ]:
def run_experiment(wording):
    """Run the agent with this wording N_TIMES times, and return every run's claims in one table."""
    tables = []
    for run in range(1, N_TIMES + 1):
        recorded.clear()
        result = langfuse.run_experiment(
            name=f"Payout tool, {wording} wording (Haiku 5.5)",
            description=f"The agent with the payout tool and the {wording} policy wording, on {MODEL}",
            data=items,
            task=lambda *, item, **kwargs: decide(agents[wording], item.input),
            evaluators=[score_claim],
            run_evaluators=[score_run],
            max_concurrency=MAX_CONCURRENCY,
            metadata={"model": MODEL, "wording": wording},
        )
        print(result.format())
        tables.append(score_table(result.item_results).assign(run=run, wording=wording))
    return pd.concat(tables, ignore_index=True)

In [ ]:
full = run_experiment("full")

In [ ]:
concise = run_experiment("concise")

## Comparing the wordings

Each confusion matrix adds up the claims from every run.

In [ ]:
show_confusion_matrix(full, "Payout tool, full wording")
show_confusion_matrix(concise, "Payout tool, concise wording")

In [ ]:
def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()


pd.DataFrame([average_scores(full), average_scores(concise)], index=["Full wording", "Concise wording"]).round(3)

The claims each wording got wrong, in any run, with the agent's rationale.

In [ ]:
wrong = pd.concat([full, concise])
wrong = wrong[~wrong["right"]]
wrong[["wording", "run", "claim_id", "handler", "agent", "handler_payout", "agent_payout", "mistake_cost", "rationale"]]

In Langfuse, open **Datasets**, then `harder-claims`, and select the two experiments to compare them claim by claim.

## Changing the cost of a wrongful decline

The cost of mistakes counts each wrongful decline at $1,500. That's a judgment about what a wrong decline costs UnderCover, in complaints, appeals and customers lost, and a different value can change which wording costs less. The runs are scored again here, with no more model calls, for values from $0 to $5,000.

In [ ]:
def cost_of_mistakes(table, wrongful_decline):
    """The cost of mistakes, averaged over the runs, with a wrongful decline at this cost."""
    costs = [mistake_cost(answer, expected, wrongful_decline) for answer, expected in zip(table["answer"], table["expected"])]
    return table.assign(cost=costs).groupby("run")["cost"].sum().mean()


values = range(0, 5001, 250)
sweep = pd.DataFrame(
    [{"wrongful_decline": value, "wording": name, "cost_of_mistakes": cost_of_mistakes(table, value)} for value in values for name, table in [("full", full), ("concise", concise)]]
)

fig = px.line(sweep, x="wrongful_decline", y="cost_of_mistakes", color="wording", markers=True, labels={"wrongful_decline": "Cost of a wrongful decline ($)", "cost_of_mistakes": "Cost of mistakes, average per run ($)"}, title="Which wording costs less, by the cost of a wrongful decline")
fig.show()

In [ ]:
by_value = sweep.pivot(index="wrongful_decline", columns="wording", values="cost_of_mistakes")
by_value["cheaper"] = by_value.apply(lambda row: "concise" if row["concise"] < row["full"] else "full" if row["full"] < row["concise"] else "same", axis=1)
by_value

## The results from Langfuse

The same comparison, read back from Langfuse, so it works without the runs in this kernel. Each row averages an experiment's runs on the whole dataset; runs on part of it, such as a `TINY` check, are left out. The lab's baseline (one run of GPT-6 Luna, without the payout tool) is there for reference.

In [ ]:
SUMMARY = ["runs", "accuracy", "decision_accuracy", "kappa", "cost_of_mistakes", "straight_through", "llm_cost", "input_tokens_a_claim", "seconds_a_claim"]


def langfuse_runs(dataset):
    """Every scored run on the whole dataset in Langfuse, one row each, with its run scores."""
    runs, cursor = [], None
    while True:
        page = langfuse.api.experiments.list(from_start_time=dataset.created_at, dataset_id=dataset.id, fields="core,scores", limit=100, cursor=cursor)
        runs += page.data
        cursor = page.meta.cursor
        if not cursor:
            break
    return pd.DataFrame([
        {"experiment": run.name.rsplit(" - ", 1)[0], "started": run.start_time, **{score.name: score.value for score in run.scores or []}}
        for run in runs
        if run.item_count == len(dataset.items) and run.scores
    ])


runs = langfuse_runs(langfuse.get_dataset(DATASET)).sort_values("started")
summary = runs.groupby("experiment", sort=False).mean(numeric_only=True).assign(runs=runs.groupby("experiment", sort=False).size())
summary[SUMMARY].round(3)

## What I found

Three runs of each wording on the 25 harder claims, on Haiku 5.5 with the payout tool. The five-claim check runs aren't counted.

### 1. Does the concise wording keep the accuracy? No.

| | Full wording | Concise wording |
|---|---|---|
| Accuracy (each run) | 0.893 (0.88, 0.92, 0.88) | 0.800 (0.76, 0.84, 0.80) |
| Decision accuracy | 0.920 | 0.907 |
| Kappa | 0.867 | 0.846 |
| Cost of mistakes, a run | $129.65 | $1,211.90 |

On accuracy and cost of mistakes, the worst full-wording run is better than the best concise one. With the concise wording the decisions are almost as good, and its second run made the best decisions of the six (0.96), but the payouts aren't. In every run it overpaid CLM-1202 by $689.95 and CLM-1216 by $328.00, and those two claims are 84% of its cost of mistakes. The payout tool does the arithmetic, but the agent chooses the lines it gets, and with the concise wording it chooses them differently, on the same claims, every time. So it's a gap in the wording, not noise. I haven't found which sentence it is: the next step is to compare the `calculate_payout` calls for those two claims in the Langfuse traces.

With both wordings the agent referred CLM-1207, which the handler approved, in every run, so that mistake doesn't come from the wording. The referrals move between claims: the full wording referred CLM-1219, which the handler declined, in all 3 runs and the concise one in 1, while the concise wording also referred CLM-1213, another decline, in 2 runs.

### 2. Is it cheaper or faster? A little, not enough to matter.

| | Full wording | Concise wording |
|---|---|---|
| Input tokens a claim | 17,065 | 14,308 (16% fewer) |
| Model cost, a run of 25 claims | $0.066 | $0.058 |
| Seconds a claim | 9.9 | 9.4 |

The concise wording has 39% fewer characters, but each claim sends only 16% fewer tokens, because every model call also carries the policy record, the claim and the tool calls so far. It saves about $0.0003 a claim in model calls and adds about $43 a claim in mistakes. Its runs are about 5% faster, but the slowest concise run (9.83 seconds a claim) is slower than the fastest full one (9.75).

### 3. Does the cost of a wrongful decline change the choice? No.

The two lines in the chart are flat. Every claim either wording declined, in any run, was one the handler declined too, so `WRONGFUL_DECLINE` never enters the cost: when the agent was unsure, it referred the claim instead. The full wording costs less for every value from $0 to $5,000, and it would for any `NEEDLESS_REFERRAL` too, since it made fewer needless referrals (6 against 7 over the three runs) and much smaller payout errors ($88.95 against $3,285.71).

### What I'd choose

The full wording. Cutting the prompt saves a fraction of a cent a claim, and one wrong payout costs hundreds of dollars. A shorter wording would have to be checked claim by claim, the approvals with payouts especially, before it replaced the full one.

Against the lab's baseline (one run of GPT-6 Luna, no payout tool: 0.52 accuracy, $3,525.90 cost of mistakes) both wordings did much better, but the model and the tool changed together, so these runs don't say which of the two helped.